# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

## ⚙️ Setup

In [1]:
%load_ext autoreload
%autoreload 2

import json, os

from _campaign_lib import *

svc = await init_services()
campaign_rounds = []
baseline_results = []

2026-03-05 20:20:33 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-05 20:20:33 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-05 20:20:33 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93


## 🎛️ Configuration

In [2]:
campaign_config = {
    "queries_per_eval": 15,              # queries per optimization evaluation step
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "pipeline_overrides": {              # Override specific pipeline params (omitted = backend default):
        # "profiling_temperature": 0.3,
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "patience": 2,                   # rounds without improvement before auto-stop
        "max_rounds": 3,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                   "descriptions to standardized database terms using entity profiling "
                   "and candidate ranking.",
        "grid_budget": 35,            # exact budget (0=full grid)
        "eval_queries_per_point": 6,  # queries per grid point (0=use all eval_data)
        "shared_queries": False,      # False=different random queries per point
        "seed": 42,
        "top_k": 5,
        "use_defaults": True,        # use DEFAULT_GRID_AXES library
    },
    "smart_search": {
        "n_diagnostic": 6,
        "max_rounds": 3,
        "stop_threshold": 0.0,
    },
}

## 📊 Data

In [3]:
# Fetch full pipeline config from backend (all parameters for reproducibility)
pipeline_raw = await svc["backend_client"].fetch_pipeline()
pipeline_config_full = pipeline_raw.get("data", pipeline_raw)
print(json.dumps(pipeline_config_full, indent=2))

# Build active step params for evaluation (from synced experiment data)
pipeline_config = load_pipeline_config(svc["exp_data"])

# Which steps to EXCLUDE from evaluation (e.g. ["llm_ranking"] for token-matching-only)
EXCLUDE_STEPS = ["llm_ranking"]

pipeline_params = build_pipeline_params(
    pipeline_config,
    overrides=campaign_config.get("pipeline_overrides"),
    exclude_steps=EXCLUDE_STEPS,
)
campaign_config["pipeline_params"] = pipeline_params
print(f"\nActive steps: {pipeline_params['steps']}")

2026-03-05 20:20:33 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "meta-llama/llama-4-maverick-17b-128e-instruct",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "short_circuit": true,
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "raw_content_limit": 5000,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
        "scrape_timeout": 5,
        "http_content_limit": 50000,
        "min_page_text_length": 200,
        "max_page_text_length": 10000,
        "title_truncate_length": 100,
        "scrape_workers": 10,
        "skip_extensions": [
          ".pdf",
          ".doc

In [4]:
#@title Backend status check
backend_status = await show_backend_status(svc["backend_client"])

2026-03-05 20:20:34 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"



BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     109
  Match Database Aliases         573
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      meta-llama/llama-4-maverick-17b-128e-instruct
  ────────────────────────────────────────────────
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings


In [5]:
#@title Dataset summary
ds_summary = show_dataset_summary(svc["store"], svc["backend_id"])


DATASET SUMMARY
  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ────────────────────────────────────────────────
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [6]:
#@title Load datasets (Excel ground truth — alternative to trace-based eval_data)
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use traces
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

if EXCEL_PATH:
    datasets = load_or_create_datasets(
        svc["store"], svc["backend_id"], EXCEL_PATH, force=FORCE_RELOAD,
    )
    train_data = datasets["train"]
    svc["session_terms"] = build_all_session_terms(svc["store"], svc["backend_id"])
    print(f"\nTrain: {len(train_data)} queries | Session terms: {len(svc['session_terms'])}")
else:
    train_data = None
    print("No EXCEL_PATH set — using trace-based eval_data (next cell).")

Loaded stored datasets: 1231 total rows
  train: 984 rows
  test_processes: 82 rows
  test_material: 165 rows

Train: 984 queries | Session terms: 94


In [7]:
#@title Langfuse — cloud sync config
# Credentials: set LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY in .env
# Project name sets the Langfuse dataset name for this campaign's eval data.
LANGFUSE_PROJECT_NAME = "termnorm_ground_truth"
LANGFUSE_BACKFILL = True  # True = push all historical runs now
LANGFUSE_RESET = False     # True = clear push state first (re-push everything)

import api.services.obs.langfuse_push as _lfp
_lfp.DATASET_NAME = LANGFUSE_PROJECT_NAME

if LANGFUSE_BACKFILL:
    if LANGFUSE_RESET:
        from api.services.obs.langfuse_push import _state_path, _fresh_state, _save_state
        _save_state(svc["store"], svc["backend_id"], _fresh_state())
        print("Langfuse push state reset — will re-push all runs.")
    n_runs = len(svc["store"].dataset_runs.list_all(svc["backend_id"]))
    if n_runs > 0:
        stats = push_langfuse(svc["store"], svc["backend_id"])
    else:
        print("No completed dataset runs yet — skipping Langfuse backfill (run after eval).")

  LANGFUSE PUSH (dataset-first)
Found 2 completed dataset runs for 'termnorm-local'
  Dataset 'termnorm_ground_truth': 6 items (0 created, 0 updated)

All 2 runs already pushed. Nothing to do.
  PUSH SUMMARY
  Total runs on disk:  2
  Newly pushed:        0
  Already done:        2
  Dataset:             termnorm_ground_truth
  Dataset items:       6


### ASSISTANT

In [8]:
baseline = load_baseline_prompt(svc["exp_data"])
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

# Excel path sets train_data; merge into eval_data
if train_data and "eval_data" not in dir():
    eval_data = train_data

print(f"Evaluation data: {len(eval_data)} queries")

Evaluation data: 984 queries


In [9]:
# OPTIONAL — skip to use random diagnostic sampling in smart search
#@title Evaluate baseline prompt
# campaign_rounds, baseline_results = await run_baseline_eval(
#     baseline, eval_data, campaign_config, svc,
# )

In [10]:
#@title Candidate coverage (post-eval diagnostic)
if baseline_results:
    _cov_data = baseline_results
else:
    # Fall back to trace-based eval data (has pipeline_data from synced experiment)
    _cov_data = load_eval_dataset(svc["store"], svc["backend_id"], svc["experiment_id"])
    _cov_data = [r for r in _cov_data if r.get("pipeline_data", {}).get("token_matched_candidates")]

if _cov_data:
    cov_df = analyze_candidate_coverage(_cov_data)
    show_entity_profiles(_cov_data)
else:
    cov_df = None
    print("No coverage data — run baseline eval or sync traces first.")

Loaded 40 eval queries
Eval runs: 2 completed runs, 1 in-progress
  run_id         name           temp  accuracy  queries
  scan_76988433  scan           0.0   33.3%     6      
  scan_c69c4424  scan           0.0   33.3%     6      
  scan_56ae503f  (in-progress)  —     —         5/?    
CANDIDATE COVERAGE
  Ground truth in candidates: 37/40 (92.5%)
  Missing from candidates:    3/40

Rank distribution (ground truth position in candidate list):
  Rank 1 (already top):  7
  Rank 2-5:              12
  Rank 6-10:             14
  Rank 11-20:            4
  Rank >20:              0
  Mean rank:             5.6
  Median rank:           5

DECISION: Coverage 92% > 50% threshold -> Reranker optimization is VIABLE.
  The ground truth exists in the candidate set; a better reranker prompt can promote it.
--- Sample 1: PMC ISO 14530-UP (GF10+MD65),M,FR Ralupol UP 804 7035.00 M/Q ---
  Core concept: molding
  Profile keys: ['entity_name', 'core_concept', 'distinguishing_features', 'key_propertie

## 🔍 Smart Search

In [11]:
#@title Task description (domain context for advisor)
# Domain context helps the advisor suggest meaningful parameter values.
# Load from file or set inline:
TASK_DESCRIPTION_PATH = r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"

if TASK_DESCRIPTION_PATH:
    from pathlib import Path
    _td_path = Path(TASK_DESCRIPTION_PATH)
    TASK_DESCRIPTION = _td_path.read_text(encoding="utf-8") if _td_path.exists() else ""
    if TASK_DESCRIPTION:
        print(f"Loaded task description: {len(TASK_DESCRIPTION)} chars from {_td_path.name}")
    else:
        print(f"Warning: {TASK_DESCRIPTION_PATH} not found")
else:
    TASK_DESCRIPTION = ""  # Or set inline: "LCA terminology normalization for ecoinvent..."

Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [12]:
#@title Scan advisor (run before sensitivity scan)
from api.models.pipeline_schema import PipelineSchema
from api.services.pipeline_discovery import TERMNORM_DEFAULT_SCHEMA

# Prefer live-enriched schema (has output_schema + prompt_meta); fall back to structural default
pipeline_schema = svc.get("pipeline_schema") or TERMNORM_DEFAULT_SCHEMA

# Filter variant library to active steps (respects EXCLUDE_STEPS)
_advisor_vl = load_variant_library()
if "pipeline_params" in campaign_config:
    from api.services.search.smart_search import filter_variant_library
    _advisor_vl = filter_variant_library(
        _advisor_vl, campaign_config["pipeline_params"], schema=pipeline_schema,
    )

# Ensure LLM client is ready
llm_client, llm_model = setup_llm(campaign_config, os.environ.get("GROQ_API_KEY", ""))

# Build coverage_stats from cov_df if available
_cov_stats = None
if "cov_df" in dir() and cov_df is not None and not cov_df.empty:
    _covered = int(cov_df["in_candidates"].sum())
    _total = len(cov_df)
    _cov_stats = {
        "covered": _covered,
        "total": _total,
        "coverage_pct": _covered / _total * 100 if _total else 0,
    }

advisory = await scan_advisor(
    pipeline_schema=pipeline_schema,
    variant_library=_advisor_vl,
    baseline_results=baseline_results,
    eval_data_size=len(train_data) if train_data else len(eval_data),
    llm_client=llm_client,
    model=llm_model,
    query_budget=120,
    coverage_stats=_cov_stats,
    excluded_steps=set(EXCLUDE_STEPS) if "EXCLUDE_STEPS" in dir() and EXCLUDE_STEPS else None,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
)

# Optionally apply the advisor's suggestion:
# campaign_config["smart_search"]["n_diagnostic"] = advisory["suggested_n_diagnostic"]

# Extract pipeline_param axes proposed by the advisor (edit in next cell)
proposed_scan_variants = advisory_to_scan_variants(advisory)
print("\n--- PROPOSED SCAN VARIANTS (edit in next cell) ---")
import pprint; pprint.pprint(proposed_scan_variants)

2026-03-05 20:20:37 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR — pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Eval data size: 984 queries
  Query budget: 120
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling meta-llama/llama-4-maverick-17b-128e-instruct ...

──────────────────────────────────────────────────────────────────────
PRIORITY AXES (ranked by importance)
──────────────────────────────────────────────────────────────────────
  1. [HIGH] profiling_schema (pipeline_param) — step: entity_profiling
     Changing the output schema directly affects downstream steps by altering the information available for candidate retrieval and ranking
     Values: [{'fields': ['entity_name', 'core_concept', 'distinguishing_features', 'key_properties', 'technical_specifications', 'alternative_names', 'classification_aliases', 'consti

In [13]:
#@title Scan variant config (edit suggested values or add your own)
# Auto-populate from advisor output, or override manually:
if "proposed_scan_variants" in dir() and proposed_scan_variants:
    scan_variants = proposed_scan_variants
else:
    scan_variants = {}

# Override / edit here:
scan_variants = {'max_token_candidates': [15, 20, 25],
 'profiling_max_tokens': [256, 512, 1024],
 'profiling_schema': [{'fields': ['entity_name',
                                  'core_concept',
                                  'distinguishing_features',
                                  'key_properties',
                                  'technical_specifications',
                                  'alternative_names',
                                  'classification_aliases',
                                  'constituent_materials',
                                  'manufacturing_processes',
                                  'applications',
                                  'notes',
                                  'material_classification']},
                      {'fields': ['entity_name',
                                  'core_concept',
                                  'distinguishing_features',
                                  'key_properties',
                                  'technical_specifications',
                                  'alternative_names',
                                  'classification_aliases',
                                  'constituent_materials',
                                  'manufacturing_processes',
                                  'applications']}],
 'profiling_temperature': [0.0, 0.3, 0.7],
 'relevance_weight_core': [0.5, 0.7, 0.9]}
print("scan_variants:", list(scan_variants.keys()) if scan_variants else "(empty)")

scan_variants: ['max_token_candidates', 'profiling_max_tokens', 'profiling_schema', 'profiling_temperature', 'relevance_weight_core']


In [14]:
#@title 🔬 Smoke test: node_overrides on /matches
# await smoke_test_override(svc, pipeline_config_full, eval_data)

In [15]:
#@title Build diagnostic set (with resume)
variant_library = load_variant_library()
if scan_variants:
    variant_library["pipeline_params"] = scan_variants
ss = campaign_config.get("smart_search", {})

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

_result = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    llm_client, llm_model,
    svc["store"], svc["backend_id"], eval_data,
    improvement_areas=campaign_config.get("improvement_areas", ""),
    variant_library=variant_library,
)
plan_id, search_baseline, diagnostic, diag_summary, cached_profiles = _result

2026-03-05 20:20:40 INFO     [api.services.search.smart_search] Building new smart search plan: ssplan_25b49c564080


  search_baseline: 110f94a84604 (render: 821 chars)


In [16]:
#@title Historical data audit
prompt_index = build_historical_index(svc["store"], svc["backend_id"])

# Try to synthesize sensitivity from grid data
if not cached_profiles:
    synth = synthesize_sensitivity(
        svc["store"], svc["backend_id"], prompt_index, diagnostic,
    )
    if synth:
        scan_df, axis_profiles = synth
        cached_profiles = axis_profiles
        print("Sensitivity derived from grid data — scan may be skippable.")

2026-03-05 20:20:41 INFO     [api.services.search.coverage] build_prompt_result_index: 2 runs -> 1 unique prompts, 6 total query results
2026-03-05 20:20:41 INFO     [api.services.search.synthesis] synthesize_sensitivity: no grid plans found


In [17]:
#@title Data inventory
inventory = show_data_inventory(prompt_index, svc["store"], svc["backend_id"])

  DATA INVENTORY  (1 prompts, 6 query results)
  Baselines: 1 plan baseline(s) — 6 queries cached

  No axis variations found in stored plans.

  Pipeline parameters (from sensitivity scans):
    max_token_candidates     3 values scanned  sensitivity: 0.000  [skip]
    profiling_schema         2 values scanned  sensitivity: 0.000  [skip]
    profiling_temperature    3 values scanned  sensitivity: 0.000  [skip]
    relevance_weight_core    3 values scanned  sensitivity: 0.000  [skip]

  Identified: 1/1 prompts (6/6 queries) via stored plans
  Unmatched:  0 prompts (0 queries)


In [18]:
#@title Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
)

  COVERAGE ADVISOR  (min_queries=6)
  Baseline: 0/6 queries cached ✗

  Prompt field axes:
    persona                4 values | 0/4 required  ✗  (0 usable, 4 uncovered)
    task_intent            3 values | 0/3 required  ✗  (0 usable, 3 uncovered)
    thinking_style         4 values | 0/4 required  ✗  (0 usable, 4 uncovered)
    answer_format          2 values | 0/2 required  ✗  (0 usable, 2 uncovered)
    problem_description    1 value  | 0/1 required  ✗  (0 usable, 1 uncovered)

  Pipeline params (always need backend):
    max_token_candidates   3 variants × 6 queries = 18 calls
    profiling_max_tokens   3 variants × 6 queries = 18 calls
    profiling_schema       2 variants × 6 queries = 12 calls
    profiling_temperature  3 variants × 6 queries = 18 calls
    relevance_weight_core  3 variants × 6 queries = 18 calls

  Summary: 0 cached, 168 still needed (84 prompt-field + 84 pipeline-param)
  >> 0/5 prompt field axes covered. Run scan to fill gaps on: persona, task_intent, thinki

In [19]:
#@title Sensitivity scan
scan_df = None
axis_profiles = []

# Detect plan status for resume
_plan_data = svc["store"].smart_search.load(svc["backend_id"], plan_id)
_plan_status = _plan_data.get("status", "") if _plan_data else ""

if cached_profiles and _plan_status in ("scan_complete", "search_complete"):
    print(f"[RESUME] Sensitivity scan already complete, "
          f"loaded {len(cached_profiles)} axis profiles")
    axis_profiles = cached_profiles
    display_axis_profiles(axis_profiles)
else:
    # Load partial checkpoint if scan was interrupted
    _partial_scan = None
    if _plan_data and _plan_status == "scan_partial":
        _sr = _plan_data.get("scan_results", {})
        _partial_scan = {
            "rows": _sr.get("rows", []),
            "completed_axes": _sr.get("completed_axes", []),
        }
        print(f"[RESUME] Found partial scan: {len(_partial_scan['completed_axes'])} axes done")

    scan_df, axis_profiles = await sensitivity_scan(
        search_baseline, variant_library, diagnostic, svc.get("backend_client"),
        user_focus=campaign_config.get("improvement_areas", ""),
        store=svc["store"], backend_id=svc["backend_id"],
        pipeline_params=campaign_config.get("pipeline_params"),
        session_terms=svc.get("session_terms"),
        plan_id=plan_id,
        prompt_result_index=prompt_index,
        partial_scan=_partial_scan,
        pipeline_schema=svc.get("pipeline_schema"),
    )

2026-03-05 20:20:41 INFO     [api.services.search.coverage] build_prompt_result_index: 2 runs -> 1 unique prompts, 6 total query results


Running sensitivity scan...
  User focus: profile schema quality, web search relevance

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Analyze entity profile and evaluate candidate matches based on core concept
    problem_description: Evaluating candidates against a given entity profile and core concept
    instruction: First, analyze the entity profile {{entity_profile_json}} to identify key featur...
    thinking_style: Think step-by-step to identify distinguishing features and prioritize candidates...
    answer_format: {"reasoning": "...", "ranked_candidates": [{"rank": 1, "candidate": "...", "rele...

  Estimated configs: ~27 x 6 queries
  Evaluating baseline...


2026-03-05 20:20:42 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


  Baseline: 2/6 (33.3%)
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.4s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 1.4s
        MISS  SJRG0010-ABS/molding                           -> Injection moulding {RoW}| injection 1.5s
        MISS  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 60 1.8s
        MISS  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 1.3s
        MISS  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 1.7s

  Axis 1/5: max_token_candidates (pipeline_param, 3 values)
  [0] 15                                         3/6  50.0%  +16.7% ^
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.4s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 1.2s
        MISS 

2026-03-05 20:21:24 INFO     [api.services.search.smart_search] Checkpoint: axis 'max_token_candidates' complete (1/5)


...

  [1] {'fields': ['entity_name', 'core_concept... 2/6  33.3%  +0.0%
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.7s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 1.6s
        MISS  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop 1.4s
        MISS  Kingfa NPG25                                   -> Polyamide (Nylon) 6.6/EU-27 1.0s
        MISS  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 1.5s
        MISS  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 1.3s

  Axis 4/5: profiling_temperature (pipeline_param, 3 values)
  [0] 0.0                                        2/6  33.3%  +0.0%
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.3s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, steel 

2026-03-05 20:22:30 INFO     [api.services.search.smart_search] Checkpoint: axis 'profiling_temperature' complete (4/5)


  [2] 0.7                                        2/6  33.3%  +0.0%
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.6s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 1.4s
        MISS  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop 1.7s
        MISS  Kingfa NPG25                                   -> Polyamide (Nylon) 6.6/EU-27 1.2s
        MISS  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 1.6s
        MISS  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 1.4s

  Axis 5/5: relevance_weight_core (pipeline_param, 3 values)
  [0] 0.5                                        3/6  50.0%  +16.7% ^
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.5s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, stee

2026-03-05 20:23:00 INFO     [api.services.search.smart_search] Checkpoint: axis 'relevance_weight_core' complete (5/5)
2026-03-05 20:23:00 INFO     [api.services.search.smart_search] Saved scan results to plan: ssplan_25b49c564080


  [2] 0.9                                        3/6  50.0%  +16.7% ^
        HIT   PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 1.5s
        HIT   Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 1.5s
        MISS  SJRG0010-ABS/molding                           -> Injection moulding {RoW}| injection 1.4s
        HIT   Kingfa NPG25                                   -> Glass fibre reinforced plastic | 75 1.5s
        MISS  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 1.8s
        MISS  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 1.4s
  >> max_token_candidates: range=16.7%, best=+16.7%, worst=+0.0%, budget=medium
  >> profiling_max_tokens: range=16.7%, best=-16.7%, worst=-33.3%, budget=medium
  >> profiling_schema: range=16.7%, best=+16.7%, worst=+0.0%, budget=low
  >> profiling_temperature: range=16.7%, best=+16.7%, worst=+0.0%, budget=

In [21]:
#@title Select best from scan & seed campaign
best_ps, best_params = select_scan_winner_notebook(
    scan_df, axis_profiles, search_baseline, variant_library,
    pipeline_params=campaign_config.get("pipeline_params"),
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

if best_params:
    campaign_config["pipeline_params"] = best_params
    print(f"Updated pipeline_params: {best_params}")

campaign_rounds.append({
    "round": "search",
    "label": f"smart_search ({best_ps.changes_description or best_ps.id[:12]})",
    "prompt_state": best_ps,
    "accuracy": campaign_rounds[0]["accuracy"] if campaign_rounds else 0.0,
    "hits": campaign_rounds[0].get("hits", 0) if campaign_rounds else 0,
    "total": campaign_rounds[0].get("total", 0) if campaign_rounds else 0,
    "results": campaign_rounds[0].get("results", []) if campaign_rounds else [],
})
display_progress(campaign_rounds)

2026-03-05 20:25:45 INFO     [api.services.search.smart_search] select_scan_winner: 0 prompt changes, 3 param changes from 3 improving axes


Selected best from 3 improving axes:
  max_token_candidates      best_delta=+16.7%  value_idx=0  acc=50.0%
  profiling_schema          best_delta=+16.7%  value_idx=0  acc=50.0%
  profiling_temperature     best_delta=+16.7%  value_idx=1  acc=50.0%
Updated pipeline_params: {'steps': ['entity_profiling', 'token_matching'], 'max_token_candidates': 15, 'profiling_schema': {'fields': ['entity_name', 'core_concept', 'distinguishing_features', 'key_properties', 'technical_specifications', 'alternative_names', 'classification_aliases', 'constituent_materials', 'manufacturing_processes', 'applications', 'notes', 'material_classification']}, 'profiling_temperature': 0.3}

Round    Accuracy   Rolling Avg    Trend
  search     0.0%         0.0%  -
  search     0.0%         0.0%  +0.0%  <-- plateau


## 🗺️ Grid Search (Optional)

<details>
<summary>Skip if you used Smart Search above. Expand for brute-force grid sweep.</summary>

**What:** Systematic sweep of the prompt configuration space (Layer 1 fields) using a cartesian product of default axis variations. Maps the accuracy landscape before hill-climbing.

**When to use:** When you want exhaustive coverage of the grid, or when Smart Search results look unreliable and you want independent validation.

**What you get:** Ranked starting points, which dimensions matter most (marginal stats), interaction effects between fields (heatmaps), and LLM-analyzed insights.

**How to read results:**
- **Ranked table** — best combos at the top; use the winner as your campaign seed
- **Marginal stats** — which axis values have the highest mean accuracy across all combos
- **Pairwise heatmaps** — green = good interaction, red = bad; look for synergies and conflicts
- **LLM analysis** — automated pattern recognition across the grid results

</details>

In [ ]:
#@title Grid campaign overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
#@title Build or resume grid plan + load eval data
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

# Load full eval_data for later optimization rounds
eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
)
if not eval_data:
    raise RuntimeError(
        "No evaluation data found. Generate data first "
        "(e.g. run evaluation.ipynb or load from DatasetStore)."
    )

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

In [ ]:
#@title Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

In [ ]:
#@title Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 🚀 Optimization

<details>
<summary>Details</summary>

Two modes:
- **Semi-automatic** (recommended): runs multiple rounds with patience-based auto-stop
- **Manual**: run one round at a time for full HITL control

Both modes subsample `eval_data` to `queries_per_eval` queries per step.

</details>

In [ ]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 💡 LLM Suggestions

<details>
<summary>Details</summary>

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 5-6.**

</details>

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 📋 Results

<details>
<summary>Details</summary>

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

</details>

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    if not base_r or not final_r:
        print("Skipping flip tracking — baseline or final results are empty.")
    else:
        flips = []
        for br, fr in zip(base_r, final_r):
            b_hit = br["hit"]
            f_hit = fr["hit"]
            if b_hit != f_hit:
                flips.append({
                    "query": br["query"][:50],
                    "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                    "base_pred": br["predicted"][:35],
                    "final_pred": fr["predicted"][:35],
                    "ground_truth": br["ground_truth"][:35],
                })

        gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
        lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

        print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
        print(f"  Queries gained (MISS->HIT): {gained}")
        print(f"  Queries lost (HIT->MISS):   {lost}")
        print(f"  Net change:                 {gained - lost:+d}")
        print()
        if flips:
            display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title 📡 Sync evaluation history to Langfuse
# Backfill: pushes any eval runs not already synced to Langfuse cloud.
# Safe to re-run — already-pushed runs are skipped automatically.
import api.services.obs.langfuse_push as _lfp
_lfp.DATASET_NAME = LANGFUSE_PROJECT_NAME
stats = push_langfuse(svc["store"], svc["backend_id"])